# Recommendation SandboxAgent

Chat directly through the SandboxAgent A2A endpoint. `chat()` opens a temporary port-forward and returns the unmodified JSON response.

Prerequisite: `kubectl` is installed and the current kube context can access the cluster.

In [3]:
import json
import subprocess
import time
import uuid
from urllib.request import Request, urlopen


def chat(agent: str, prompt: str, session: str | None = None):
    session = session or str(uuid.uuid4())
    message_id = str(uuid.uuid4())
    tunnel = subprocess.Popen(
        ["kubectl", "-n", "kagent", "port-forward",
         "service/kagent-controller", "18084:8083"],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
    )
    try:
        time.sleep(2)
        if tunnel.poll() is not None:
            raise RuntimeError(tunnel.stderr.read().strip())
        payload = {
            "jsonrpc": "2.0", "id": message_id, "method": "message/send",
            "params": {"message": {"kind": "message", "messageId": message_id,
                "role": "user", "contextId": session,
                "parts": [{"kind": "text", "text": prompt}]}},
        }
        request = Request(
            f"http://127.0.0.1:18084/api/a2a-sandboxes/kagent/{agent}/",
            data=json.dumps(payload).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urlopen(request, timeout=300) as response:
            return json.load(response)
    finally:
        tunnel.terminate()
        try:
            tunnel.wait(timeout=5)
        except subprocess.TimeoutExpired:
            tunnel.kill()

In [4]:
result = chat(
    "recsys-recommendation-agent-sandbox",
    "Gợi ý 3 sản phẩm phù hợp cho user 1001.",
)

result

{'jsonrpc': '2.0',
 'id': '2c798021-beed-4f08-899b-b194defb2764',
 'result': {'kind': 'task',
  'id': '01a0382b-6d82-7585-8ada-30c0d381d91b',
  'artifacts': [{'artifactId': '01a0382b-f0b8-7b21-b82d-f16babd5bf1e',
    'parts': [{'kind': 'text',
      'text': 'Dựa trên kết quả phân tích, với người dùng số 1001, tôi có thể gợi ý 3 sản phẩm phù hợp nhất:\n\n1. **Sản phẩm 1**: Item ID: 800075, Score: 0.5806801915168762\n2. **Sản phẩm 2**: Item ID'}]}],
  'contextId': '683479a7-e5f3-4b75-9fa1-987594f12d09',
  'history': [{'kind': 'message',
    'messageId': '2c798021-beed-4f08-899b-b194defb2764',
    'contextId': '683479a7-e5f3-4b75-9fa1-987594f12d09',
    'parts': [{'kind': 'text',
      'text': 'Gợi ý 3 sản phẩm phù hợp cho user 1001.'}],
    'role': 'user'},
   {'kind': 'message',
    'messageId': '2c798021-beed-4f08-899b-b194defb2764',
    'contextId': '683479a7-e5f3-4b75-9fa1-987594f12d09',
    'parts': [{'kind': 'text',
      'text': 'Gợi ý 3 sản phẩm phù hợp cho user 1001.'}],
    'ro

### Conclusion from the recorded result

The result above confirms that `recsys-recommendation-agent-sandbox` invoked MCP to execute a function instead of generating the recommendation list directly with the LLM:

- `result.history` contains an `adk_type: function_call` event for `get_personalized_recommendations` with `user_id=1001`, `candidate_item_ids=None`, and `top_k=3`.
- The following `adk_type: function_response` event has the same tool-call ID, `m2tl7GdRVXjCV34Xis4j9M8XgFWv1qQX`, and returns items `800075`, `800145`, and `800015` with `model_version=20260823155004`.
- `status.state: completed` confirms that the A2A task finished. The paired `function_call` and `function_response` directly demonstrate the `SandboxAgent -> RemoteMCPServer -> get_personalized_recommendations` execution path.
- The final presentation text was truncated at the generation limit (`candidatesTokenCount=256`), but the MCP response already contains all three products. This is not a tool-execution failure.

The notebook intentionally displays the raw A2A response so that the execution trace remains intact, without a parser or intermediary helper.